In [45]:
import gradio as gr 

history = [{"role" : "assistant", "content" : "어서오세요! Hire Me 입니다.\n취업 분석을 원하시면 이력서를 업로드 해보세요.😀"}, {"role" : "user", "content" : "안녕 나는 평생 직장을 찾고 있어"}]

theme = gr.themes.Soft(
    primary_hue="gray",
    secondary_hue="stone",
    neutral_hue="zinc",
    font=[gr.themes.GoogleFont('Noto Sans Korean '), gr.themes.GoogleFont('42dot Sans'), gr.themes.GoogleFont('Nanum Gothic '), 'sans-serif'],
).set(
    button_transform_hover='*button_primary_background_fill'
)

# CSS 스타일 정의
custom_css = """ 
.custom-row {
    display: block;
}

.custom-button {
    margin: 10px; /* 버튼 주변 여백 제거 */
    padding: 5px 5px; /* 버튼 내부 여백 조정 */
    height: 40px; /* 버튼 높이 고정 */
    width : 40px;
    display: inline-flex; /* 수평 정렬 */
    align-items: center; /* 수직 가운데 정렬 */
    justify-content: center;
}

.custom-input {    
    align-items: top; /* 수직 가운데 정렬 */      
}

.custom-checkbox {
    display: flex; justify-content: flex-end;
    margin : 0px;        
    }

.gr-avatar {
    width: 40px !important;  /* 아바타 너비 */
    height: 40px !important; /* 아바타 높이 */
    border-radius: 50%; /* 동그란 모양 유지 */
}    
"""

from functools import partial

with gr.Blocks(theme=theme, css=custom_css) as demo :

    chat_history = gr.State([])

    # gradio 기본기능인 exmaple 기능 사용, 이 함수에는 evt 외의 추가 인수를 넣을 수 없음 
    def handle_example_click(evt: gr.SelectData):
        """ llm 함수 실행 """
        result = "좋은 질문이에요! 🤖 지금 분석 중입니다."
        example_history = [{"role": "user", "content": evt.value['text']}, {"role": "assistant", "content": result}]
        return example_history
        
    # 사용자 입력 처리 함수
    def handle_user_message(user_msg, chat_history):
        chat_history = chat_history or []
        chat_history.append({"role": "user", "content": user_msg})
        assistant_response = "좋은 질문이에요! 🤖 지금 분석 중입니다."
        chat_history.append({"role": "assistant", "content": assistant_response})
        return chat_history, chat_history  # 하나는 Chatbot용, 하나는 State용
    
    def clear_user_input() :
        return gr.update(value=None)

    examples = [{"text" : "프론트엔드 쪽은 어떤 회사들이 채용 중이야?"}, {"text" : "종로서 일할 수 있는 개발자 공고 보여줘"}, {"text" : "내가 개선할 수 있는 부분을 알려줘"}, {"text" : "나에게 잘 맞는 직무를 찾고 싶어"}]
            
    ### 챗봇            
    chatbot = gr.Chatbot(height="85vh", type='messages', container=False, avatar_images=("chicken.png", "magicball.jpeg"), examples=examples)
    # placeholder="어서오세요! Hire Me 입니다. 취업 분석을 원하시면 이력서를 업로드 해보세요.😀"
    # examples=[{"text": "취업할만한 회사를 찾고 싶어"}, {"text": "내 커리어 로드맵을 그리고 싶어"}]
    
    with gr.Row() :
        gr.UploadButton(label="upload", icon="img.png", file_types=["image"], scale=1, elem_classes=["custom-button"])
        with gr.Column(elem_classes=["custom-input"], scale=30) :
            user_input = gr.Textbox(placeholder="무엇이든 물어보세요", show_label=False, scale=30)
            gr.Checkbox(label="🪄 관련 강의 추천 받기", elem_classes=["custom-checkbox"])
        submit_btn = gr.Button("🚀 Send", scale=1, size="md", elem_classes=["custom-button"])
    
    with gr.Sidebar(open=False) :
        gr.Markdown("Profile")
        gr.Image(value="profile.png", width="200px", show_label=False, show_download_button=False, show_fullscreen_button=False, show_share_button=False)

        # 버튼 개별화 : for문으로 압축해보려 했는데, 압축시 동적함수 생성 필요 -> gradio 자체에서 막혀있음, 우회 필요 -> 그냥 간단하게 개별로 리스너 연결하기로 함        
        btn_job = gr.Button("🎯 나에게 맞는 직무는?", value="나에게 맞는 직무를 추천해줘", size="sm")
        btn_roadmap = gr.Button("📍 커리어 로드맵 그리기", value="나에게 맞는 커리어 로드맵을 만들어줘", size="sm")
        btn_company = gr.Button("🔍 인생 회사 찾기", value="나에게 맞는 공고를 추천해줘", size="sm")        
        btn_weakness = gr.Button("🔥 내가 보완할 부분은?", value="채용 공고들과 커리어로드맵을 고려할 때 내가 보완해야할 부분은 뭘까?", size="sm")
        
        btn_save_chat = gr.DownloadButton("📂 대화 내용 저장하기", value="filepath", variant="huggingface", size="sm")

        # 소개 및 푸터
        gr.Markdown("🐚 ABOUT US")
        gr.Markdown("서비스 소개 @@@@")

        gr.Image(value="cherry.png", width="250px", show_label=False, show_download_button=False, show_fullscreen_button=False, show_share_button=False)

    chatbot.example_select(handle_example_click, outputs=[chatbot])
    # 사용자 질문 제출 처리
    submit_btn.click(fn=handle_user_message, inputs=[user_input, chatbot], outputs=[chatbot, chat_history]).then(clear_user_input, None, user_input)
    
    btn_job.click(handle_user_message, inputs=[btn_job, chat_history], outputs=[chatbot, chat_history])
    btn_roadmap.click(handle_user_message, inputs=[btn_roadmap, chat_history], outputs=[chatbot, chat_history])
    btn_company.click(handle_user_message, inputs=[btn_company, chat_history], outputs=[chatbot, chat_history])
    btn_weakness.click(handle_user_message, inputs=[btn_weakness, chat_history], outputs=[chatbot, chat_history])


demo.launch()


* Running on local URL:  http://127.0.0.1:7893

To create a public link, set `share=True` in `launch()`.


In [41]:
import gradio as gr

examples = [{"text" : "프론트엔드 쪽은 어떤 회사들이 채용 중이야?"}, {"text" : "종로서 일할 수 있는 개발자 공고 보여줘"}]

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(height=300, avatar_images=("chicken.png", "magicball.jpeg"), examples=examples)

    # 예제 클릭 시 실행할 함수 정의
    def handle_example_click(evt: gr.SelectData):
        result = "좋은 질문이에요! 🤖 지금 분석 중입니다."
        example_history = [{"role": "user", "content": evt.value['text']}, {"role": "assistant", "content": result}]
        return [[evt.value['text'], "좋은 질문이에요! 🤖 지금 분석 중입니다."]]

    # 예제 클릭 이벤트 핸들링
    chatbot.example_select(handle_example_click, outputs=[chatbot])

demo.launch()

C:\Users\volav\AppData\Local\Temp\ipykernel_19788\2773672707.py:6: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=300, avatar_images=("chicken.png", "magicball.jpeg"), examples=examples)


* Running on local URL:  http://127.0.0.1:7889

To create a public link, set `share=True` in `launch()`.
